# Fire analysis


In [1]:
DB_FILE = "fire.duckdb"
STATE = "OR"

## DuckDB / SQL Magic setup

Using [JupySQL](https://jupysql.readthedocs.io/). [Docs from DuckDB.](https://duckdb.org/docs/current/guides/python/jupyter)


In [2]:
%config SqlMagic.autopandas = True
%config SqlMagic.displaycon = False

In [3]:
import duckdb

%load_ext sql
conn = duckdb.connect(DB_FILE)
%sql conn --alias duckdb

Tip: You may define configurations in /Users/afeld/dev/ptps-wildfire-demo/pyproject.toml or /Users/afeld/.jupysql/config.

Did not find user configurations in /Users/afeld/dev/ptps-wildfire-demo/pyproject.toml.

In [4]:
from helpers import run_script_in_db


run_script_in_db(conn, "setup.sql")
run_script_in_db(conn, "views.sql")

In [5]:
import pandas as pd

pd.set_option("display.max_columns", 100)

## Burn probabilities


In [6]:
%%sql
SELECT DISTINCT state.stusps
FROM state_boundaries AS state
INNER JOIN burn_prob_1km AS bp
    ON ST_Contains(state.geom, bp.point)
ORDER BY state.stusps;

,STUSPS
0,CA
1,OR
2,WA


In [7]:
%%sql
SELECT
    min(bp_2011),
    max(bp_2011)
FROM burn_prob_1km;

,min(bp_2011),max(bp_2011)
0,0.0,0.017974


In [8]:
from lonboard import Map, ScatterplotLayer

from helpers import to_continuous_color_map

query = """
SELECT
    point,
    bp_2011
FROM burn_prob_1km
WHERE bp_2011 > 0.0;
"""

values = conn.execute(query).df()["bp_2011"]
colors = to_continuous_color_map(values, "OrRd")

layer = ScatterplotLayer.from_duckdb(
    query,
    conn,
    crs="EPSG:4326",
    get_fill_color=colors,
    radius_min_pixels=2,
    radius_scale=1000,
    radius_units="meters",
)
m = Map(layer)
m

## State-level


In [9]:
%%sql
SELECT COUNT(*)
FROM burn_prob_1km AS bp
INNER JOIN state_boundaries AS state
    ON ST_Contains(state.geom, bp.point)
WHERE state.stusps = '{{STATE}}';

,count_star()
0,2599


In [10]:
query = f"""
SELECT
    bp.point,
    bp.bp_2011
FROM burn_prob_1km AS bp
INNER JOIN state_boundaries AS state
    ON ST_Contains(state.geom, bp.point)
WHERE state.stusps = '{STATE}'
AND bp_2011 > 0.0;
"""

values = conn.execute(query).df()["bp_2011"]
colors = to_continuous_color_map(values, "OrRd")

layer = ScatterplotLayer.from_duckdb(
    query,
    conn,
    crs="EPSG:4326",
    get_fill_color=colors,
    radius_min_pixels=2,
    radius_scale=1000,
    radius_units="meters",
)
m = Map(layer)
m

## Red flag alerts


In [12]:
raw_alerts = get_geo_df(
    conn,
    """
    SELECT *
    FROM red_flag_alerts
    """,
)
raw_alerts

,OGC_FID,@id,@type,id,areaDesc,geocode,affectedZones,references,sent,effective,onset,expires,ends,status,messageType,category,severity,certainty,urgency,event,sender,senderName,headline,description,instruction,response,note,parameters,scope,code,language,web,eventCode,geom
0,0,https://api.weather.gov/alerts/urn:oid:2.49.0....,wx:Alert,urn:oid:2.49.0.1.840.0.66a40e778a67f4e13c3f34d...,Fall River County Area; Northern Campbell; Sou...,"{ ""SAME"": [ ""046033"", ""046047"", ""056005"", ""056...","[https://api.weather.gov/zones/fire/SDZ322, ht...",[ ],2026-08-09 21:14:00-04:00,2026-08-09 21:14:00-04:00,2026-08-10 08:00:00-04:00,2026-08-10 09:15:00-04:00,2026-08-10 17:00:00-04:00,Actual,Alert,Met,Severe,Likely,Expected,Red Flag Warning,w-nws.webmaster@noaa.gov,NWS Rapid City SD,Red Flag Warning issued August 10 at 1:14AM MD...,...RED FLAG WARNING IN EFFECT FROM NOON TODAY ...,A Red Flag Warning means that critical fire we...,Prepare,None,"{ ""AWIPSidentifier"": [ ""RFWUNR"" ], ""WMOidentif...",Public,IPAWSv1.0,en-US,http://www.weather.gov,"{ ""SAME"": [ ""NWS"" ], ""NationalWeatherService"":...",None
1,1,https://api.weather.gov/alerts/urn:oid:2.49.0....,wx:Alert,urn:oid:2.49.0.1.840.0.88b8fc20efce9f253fe8857...,Niobrara/Lower Elevations of Converse/Thunder ...,"{ ""SAME"": [ ""056009"", ""056027"", ""056015"", ""056...","[https://api.weather.gov/zones/fire/WYZ417, ht...",[ ],2026-08-09 21:03:00-04:00,2026-08-09 21:03:00-04:00,2026-08-10 08:00:00-04:00,2026-08-10 17:00:00-04:00,2026-08-10 17:00:00-04:00,Actual,Alert,Met,Severe,Likely,Expected,Red Flag Warning,w-nws.webmaster@noaa.gov,NWS Cheyenne WY,Red Flag Warning issued August 10 at 1:03AM MD...,The National Weather Service in Cheyenne has i...,A Red Flag Warning means that critical fire we...,Prepare,None,"{ ""AWIPSidentifier"": [ ""RFWCYS"" ], ""WMOidentif...",Public,IPAWSv1.0,en-US,http://www.weather.gov,"{ ""SAME"": [ ""NWS"" ], ""NationalWeatherService"":...",None
2,2,https://api.weather.gov/alerts/urn:oid:2.49.0....,wx:Alert,urn:oid:2.49.0.1.840.0.a4691733318da694a6d5792...,East Salmon River Mountains/Salmon NF; Lemhi a...,"{ ""SAME"": [ ""016037"", ""016049"", ""016059"", ""016...","[https://api.weather.gov/zones/fire/IDZ475, ht...",[ ],2026-08-09 17:37:00-04:00,2026-08-09 17:37:00-04:00,2026-08-10 10:00:00-04:00,2026-08-10 11:00:00-04:00,2026-08-10 17:00:00-04:00,Actual,Alert,Met,Severe,Likely,Expected,Red Flag Warning,w-nws.webmaster@noaa.gov,NWS Pocatello ID,Red Flag Warning issued August 9 at 9:37PM MDT...,The National Weather Service in Pocatello has ...,A Red Flag Warning means that critical fire we...,Prepare,None,"{ ""AWIPSidentifier"": [ ""RFWPIH"" ], ""WMOidentif...",Public,IPAWSv1.0,en-US,http://www.weather.gov,"{ ""SAME"": [ ""NWS"" ], ""NationalWeatherService"":...",None
3,3,https://api.weather.gov/alerts/urn:oid:2.49.0....,wx:Alert,urn:oid:2.49.0.1.840.0.a4691733318da694a6d5792...,Upper Snake River Valley/Idaho Falls BLM; Midd...,"{ ""SAME"": [ ""016005"", ""016011"", ""016013"", ""016...","[https://api.weather.gov/zones/fire/IDZ410, ht...",[ ],2026-08-09 17:37:00-04:00,2026-08-09 17:37:00-04:00,2026-08-10 10:00:00-04:00,2026-08-10 11:00:00-04:00,2026-08-10 17:00:00-04:00,Actual,Alert,Met,Severe,Likely,Expected,Red Flag Warning,w-nws.webmaster@noaa.gov,NWS Pocatello ID,Red Flag Warning issued August 9 at 9:37PM MDT...,The National Weather Service in Pocatello has ...,A Red Flag Warning means that critical fire we...,Prepare,None,"{ ""AWIPSidentifier"": [ ""RFWPIH"" ], ""WMOidentif...",Public,IPAWSv1.0,en-US,http://www.weather.gov,"{ ""SAME"": [ ""NWS"" ], ""NationalWeatherService"":...",None


### Geometries

[The alerts/zones APIs don't include the geometry](https://github.com/weather-gov/api/discussions/278), so backfill them.


In [11]:
import requests
import shapely

from helpers import get_geo_df

zones = get_geo_df(
    conn,
    """
    SELECT DISTINCT
        unnest(affectedZones) AS zone_url,
        geom
    FROM red_flag_alerts
    """,
)


def get_zone_geom(url: str):
    response = requests.get(url)
    geom = response.json()["geometry"]
    return shapely.geometry.shape(geom)


zone_urls = zones["zone_url"]
zones = zones.assign(geom=zone_urls.apply(get_zone_geom))  # pyright: ignore[reportArgumentType, reportCallIssue]
zones

,zone_url,geom
0,https://api.weather.gov/zones/fire/IDZ475,"POLYGON ((-113.9809 45.70271, -113.9736 45.702..."
1,https://api.weather.gov/zones/fire/WYZ420,"POLYGON ((-106.03059 42.44971, -106.03058 42.4..."
2,https://api.weather.gov/zones/fire/IDZ476,"POLYGON ((-113.88463 45.38238, -113.84451 45.3..."
3,https://api.weather.gov/zones/fire/IDZ410,"POLYGON ((-111.55316 44.4876, -111.55382 44.47..."
4,https://api.weather.gov/zones/fire/WYZ417,"POLYGON ((-104.05479 43.50311, -104.05479 43.4..."
5,https://api.weather.gov/zones/fire/WYZ315,"POLYGON ((-105.0939 43.49831, -105.126 43.4979..."
6,https://api.weather.gov/zones/fire/WYZ427,"POLYGON ((-106.1982 41.86051, -106.18639 41.86..."
7,https://api.weather.gov/zones/fire/WYZ421,"POLYGON ((-107.2771 42.43471, -107.0006 42.433..."
8,https://api.weather.gov/zones/fire/WYZ424,"POLYGON ((-107.0891 41.34751, -107.0888 41.341..."
9,https://api.weather.gov/zones/fire/WYZ426,"POLYGON ((-106.51842 41.67727, -106.51 41.6697..."


In [ ]:
def get_alert_geom(row: pd.Series):
    """Returns the unified geometry for all affected zones"""

    alert_zones = zones[zones["zone_url"].isin(row["affectedZones"])]
    return alert_zones.union_all("unary")


alerts = raw_alerts.assign(geom=raw_alerts.apply(get_alert_geom, axis=1))  # pyright: ignore[reportArgumentType]
alerts

,OGC_FID,@id,@type,id,areaDesc,geocode,affectedZones,references,sent,effective,onset,expires,ends,status,messageType,category,severity,certainty,urgency,event,sender,senderName,headline,description,instruction,response,note,parameters,scope,code,language,web,eventCode,geom
0,0,https://api.weather.gov/alerts/urn:oid:2.49.0....,wx:Alert,urn:oid:2.49.0.1.840.0.66a40e778a67f4e13c3f34d...,Fall River County Area; Northern Campbell; Sou...,"{ ""SAME"": [ ""046033"", ""046047"", ""056005"", ""056...","[https://api.weather.gov/zones/fire/SDZ322, ht...",[ ],2026-08-09 21:14:00-04:00,2026-08-09 21:14:00-04:00,2026-08-10 08:00:00-04:00,2026-08-10 09:15:00-04:00,2026-08-10 17:00:00-04:00,Actual,Alert,Met,Severe,Likely,Expected,Red Flag Warning,w-nws.webmaster@noaa.gov,NWS Rapid City SD,Red Flag Warning issued August 10 at 1:14AM MD...,...RED FLAG WARNING IN EFFECT FROM NOON TODAY ...,A Red Flag Warning means that critical fire we...,Prepare,None,"{ ""AWIPSidentifier"": [ ""RFWUNR"" ], ""WMOidentif...",Public,IPAWSv1.0,en-US,http://www.weather.gov,"{ ""SAME"": [ ""NWS"" ], ""NationalWeatherService"":...","POLYGON ((-103.37019 43.47761, -103.36461 43.0..."
1,1,https://api.weather.gov/alerts/urn:oid:2.49.0....,wx:Alert,urn:oid:2.49.0.1.840.0.88b8fc20efce9f253fe8857...,Niobrara/Lower Elevations of Converse/Thunder ...,"{ ""SAME"": [ ""056009"", ""056027"", ""056015"", ""056...","[https://api.weather.gov/zones/fire/WYZ417, ht...",[ ],2026-08-09 21:03:00-04:00,2026-08-09 21:03:00-04:00,2026-08-10 08:00:00-04:00,2026-08-10 17:00:00-04:00,2026-08-10 17:00:00-04:00,Actual,Alert,Met,Severe,Likely,Expected,Red Flag Warning,w-nws.webmaster@noaa.gov,NWS Cheyenne WY,Red Flag Warning issued August 10 at 1:03AM MD...,The National Weather Service in Cheyenne has i...,A Red Flag Warning means that critical fire we...,Prepare,None,"{ ""AWIPSidentifier"": [ ""RFWCYS"" ], ""WMOidentif...",Public,IPAWSv1.0,en-US,http://www.weather.gov,"{ ""SAME"": [ ""NWS"" ], ""NationalWeatherService"":...","POLYGON ((-106.494 41.00211, -106.5006 41.0019..."
2,2,https://api.weather.gov/alerts/urn:oid:2.49.0....,wx:Alert,urn:oid:2.49.0.1.840.0.a4691733318da694a6d5792...,East Salmon River Mountains/Salmon NF; Lemhi a...,"{ ""SAME"": [ ""016037"", ""016049"", ""016059"", ""016...","[https://api.weather.gov/zones/fire/IDZ475, ht...",[ ],2026-08-09 17:37:00-04:00,2026-08-09 17:37:00-04:00,2026-08-10 10:00:00-04:00,2026-08-10 11:00:00-04:00,2026-08-10 17:00:00-04:00,Actual,Alert,Met,Severe,Likely,Expected,Red Flag Warning,w-nws.webmaster@noaa.gov,NWS Pocatello ID,Red Flag Warning issued August 9 at 9:37PM MDT...,The National Weather Service in Pocatello has ...,A Red Flag Warning means that critical fire we...,Prepare,None,"{ ""AWIPSidentifier"": [ ""RFWPIH"" ], ""WMOidentif...",Public,IPAWSv1.0,en-US,http://www.weather.gov,"{ ""SAME"": [ ""NWS"" ], ""NationalWeatherService"":...","POLYGON ((-113.68824 45.25922, -113.6879 45.25..."
3,3,https://api.weather.gov/alerts/urn:oid:2.49.0....,wx:Alert,urn:oid:2.49.0.1.840.0.a4691733318da694a6d5792...,Upper Snake River Valley/Idaho Falls BLM; Midd...,"{ ""SAME"": [ ""016005"", ""016011"", ""016013"", ""016...","[https://api.weather.gov/zones/fire/IDZ410, ht...",[ ],2026-08-09 17:37:00-04:00,2026-08-09 17:37:00-04:00,2026-08-10 10:00:00-04:00,2026-08-10 11:00:00-04:00,2026-08-10 17:00:00-04:00,Actual,Alert,Met,Severe,Likely,Expected,Red Flag Warning,w-nws.webmaster@noaa.gov,NWS Pocatello ID,Red Flag Warning issued August 9 at 9:37PM MDT...,The National Weather Service in Pocatello has ...,A Red Flag Warning means that critical fire we...,Prepare,None,"{ ""AWIPSidentifier"": [ ""RFWPIH"" ], ""WMOidentif...",Public,IPAWSv1.0,en-US,http://www.weather.gov,"{ ""SAME"": [ ""NWS"" ], ""NationalWeatherService"":...","POLYGON ((-112.87259 42.77726, -112.87777 42.7..."


### Map


In [14]:
from lonboard import PolygonLayer

layer = PolygonLayer.from_geopandas(
    alerts,
    # crs="EPSG:4326",
    get_fill_color=[255, 0, 0],
)
m = Map(layer)
m

/Users/afeld/dev/ptps-wildfire-demo/.venv/lib/python3.14/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


## Active fires


### Query data

> Each MODIS active fire/thermal hotspot location represents the center of a 1km pixel that is flagged by the algorithm as containing one or more fires within the pixel.

https://www.earthdata.nasa.gov/data/tools/firms

https://firms.modaps.eosdis.nasa.gov/active_fire/#firms-txt


In [15]:
%%sql
SELECT *
FROM active_fires
LIMIT 10;


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,confidence,version,bright_t31,frp,daynight
0,45.29657,-69.49461,326.35,1.42,1.18,2026-08-09,0026,T,100,6.1NRT,286.45,41.08,N
1,18.31891,-88.33073,316.88,2.35,1.48,2026-08-09,0158,T,92,6.1NRT,293.69,42.97,N
2,18.31735,-88.33570,314.31,2.35,1.48,2026-08-09,0158,T,88,6.1NRT,293.83,36.78,N
3,20.72093,-76.02602,306.44,1.95,1.36,2026-08-09,0158,T,45,6.1NRT,290.02,14.14,N
4,19.39528,-92.03804,306.85,4.28,1.91,2026-08-09,0158,T,49,6.1NRT,292.19,36.88,N
5,19.40042,-92.04571,305.32,4.28,1.91,2026-08-09,0158,T,27,6.1NRT,292.26,30.11,N
6,20.54469,-87.33319,314.37,1.78,1.31,2026-08-09,0158,T,89,6.1NRT,292.90,25.21,N
7,21.53366,-87.03705,305.53,1.62,1.25,2026-08-09,0158,T,54,6.1NRT,294.12,8.01,N
8,21.54455,-87.03920,328.75,1.62,1.25,2026-08-09,0158,T,100,6.1NRT,294.33,49.83,N
9,41.45987,-81.67714,302.94,1.55,1.23,2026-08-09,0204,T,52,6.1NRT,290.15,11.42,N
